# Manisha Overseas — One-click AI Virtual Try-On (Research Only)

**Prepared for your Stage 4 internal test.** The single code cell below automatically verifies your GPU, installs FASHN VTON v1.5 if needed, downloads model weights, loads a Manisha Overseas women's dress and an existing female-model catalogue photograph, generates a research-only AI try-on, and displays the original and generated images. **You do not need to run Sections 2–7 separately.**

You must still have a free GPU connected in Google Colab, and you must click the code cell's ▶ button once. I cannot press that button from ChatGPT. **Do not choose paid hardware or pay for Hugging Face.**

- Default garment: **Style 4638**. Female-model reference: **Style 3476**. To change the garment after generation, use the more detailed existing notebook.
- **Internal research/evaluation only.** The model's standard human-parser dependency inherits NVIDIA non-commercial restrictions. Don't publish AI try-ons on the commercial site before proper licensing.
- **Model photograph permissions:** Confirm the company has permission to create AI-edited imagery featuring the catalogue model.
- No customer uploads, API tokens, personal photographs, or GitHub photo uploads. Colab runtime data are handled under Google's policies; use test assets only.
- The model can change face/body appearance or prints. Results are not sizing or fit guarantees. The free T4 may encounter VRAM limits.

**The GitHub live `main` branch and your website are untouched.**

In [ ]:
#========================================================================
# AUTOMATED STEP 1 / 8
#========================================================================
import torch
from IPython.display import display, Markdown

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab: Runtime → Change runtime type → GPU. Free GPU availability is not guaranteed.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (gpu.total_memory / (1024**3)))
print("CUDA capability:", f"{gpu.major}.{gpu.minor}")
if gpu.major < 8:
    print("WARNING: This older GPU may need float32 and run out of VRAM. It may run out of GPU memory during generation; no purchase is necessary to diagnose this.")
display(Markdown("✅ GPU runtime detected. **No payment is required to run on the free tier if one is allocated.**"))

#========================================================================
# AUTOMATED STEP 2 / 8
#========================================================================
# One clean installation cell — run this before model weights or generation.
from pathlib import Path
import sys
import subprocess
import importlib

ROOT = Path("/content")
REPO = ROOT / "fashn-vton-1.5"
UPSTREAM = "https://github.com/fashn-AI/fashn-vton-1.5.git"

print("STEP 2 / Installing official open-source FASHN VTON package...")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", UPSTREAM, str(REPO)], check=True)
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError(
        f"Expected official project at {REPO}, but pyproject.toml is missing. "
        "Delete that incomplete folder in Colab Files and rerun this cell."
    )

# Install in the SAME Python environment used by this Colab notebook.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-e", str(REPO)],
    check=True
)
# Editable installs sometimes need a kernel restart in Colab.
# Add the official src-layout path for this current notebook kernel.
source_dir = str(REPO / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
try:
    from fashn_vton import TryOnPipeline
except Exception as e:
    raise RuntimeError(
        "The package installation finished, but import still failed. "
        "Use Runtime → Restart session, then Runtime → Run all. "
        f"Original import error: {type(e).__name__}: {e}"
    ) from e
print("✅ INSTALLATION VERIFIED: fashn_vton imports correctly.")
print("Official project folder:", REPO)

#========================================================================
# AUTOMATED STEP 3 / 8
#========================================================================
# Run after STEP 2 completes successfully.
import sys
import subprocess
from pathlib import Path

if "REPO" not in globals() or not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("First run Section 2 — Install the official FASHN VTON repository.")

WEIGHTS = REPO / "weights"
required = [
    WEIGHTS / "model.safetensors",
    WEIGHTS / "dwpose" / "yolox_l.onnx",
    WEIGHTS / "dwpose" / "dw-ll_ucoco_384.onnx"
]
if not all(p.is_file() and p.stat().st_size > 0 for p in required):
    print("STEP 3 / Downloading official model and pose weights (large files)...")
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "download_weights.py"),
         "--weights-dir", str(WEIGHTS)],
        check=True
    )
missing = [str(p) for p in required if not (p.is_file() and p.stat().st_size > 0)]
if missing:
    raise RuntimeError("Weight download incomplete: " + ", ".join(missing))
print("✅ WEIGHTS VERIFIED. All three required files exist.")
print("Human parser weights may be fetched automatically during first inference.")

#========================================================================
# AUTOMATED STEP 4 / 8
#========================================================================
from io import BytesIO
import requests
from PIL import Image, ImageOps
from IPython.display import display

STYLES = {
    "4638": ("Floral printed long dress", "style-4638-front.webp"),
    "6821": ("Printed wide-leg one-piece", "style-6821-front.webp"),
    "3476": ("Printed long dress", "style-3476-front.webp"),
    "7284": ("Open-back long dress", "style-7284-front.webp"),
    "1509": ("Tiered long dress", "style-1509-front.webp"),
    "2941": ("Printed tiered dress", "style-2941-front.webp"),
    "9152": ("Green printed long dress", "style-9152-full.webp"),
}
STYLE_CODE = "4638"  # Change this to "3476", "9152", etc.

if STYLE_CODE not in STYLES:
    raise ValueError(f"Unknown style {STYLE_CODE}. Choose from: {', '.join(STYLES)}")
label, filename = STYLES[STYLE_CODE]
url = (
    "https://raw.githubusercontent.com/"
    "shubhampuri240/manisha-overseas-website/staging/assets/visuals/" + filename
)
with requests.get(url, timeout=45, stream=True) as response:
    response.raise_for_status()
    data = bytearray()
    for chunk in response.iter_content(chunk_size=65536):
        data.extend(chunk)
        if len(data) > 15 * 1024 * 1024:
            raise ValueError("Garment file larger than the 15 MB evaluation limit.")
with Image.open(BytesIO(data)) as im:
    if im.width * im.height > 30_000_000:
        raise ValueError("Garment photograph exceeds 30 megapixels.")
    garment = ImageOps.exif_transpose(im).convert("RGB").copy()
print(f"Selected Manisha Overseas style {STYLE_CODE}: {label}")
display(garment.resize((int(garment.width * min(420/garment.width, 1)), int(garment.height * min(420/garment.width, 1)))))

#========================================================================
# AUTOMATED STEP 5 / 8
#========================================================================
# Default test person: an existing female fashion model from our own catalogue.
# No upstream male demo person; no private/customer upload.
from io import BytesIO
from PIL import Image, ImageOps
import requests
from IPython.display import display

if "STYLE_CODE" not in globals() or "STYLES" not in globals():
    raise RuntimeError("Run Section 4 first to select a Manisha Overseas garment.")

PERSON_REFERENCE_STYLE = "4638" if STYLE_CODE == "3476" else "3476"
_, reference_filename = STYLES[PERSON_REFERENCE_STYLE]
person_url = (
    "https://raw.githubusercontent.com/"
    "shubhampuri240/manisha-overseas-website/staging/assets/visuals/"
    + reference_filename
)
with requests.get(person_url, timeout=45, stream=True) as response:
    response.raise_for_status()
    payload = bytearray()
    for chunk in response.iter_content(chunk_size=65536):
        payload.extend(chunk)
        if len(payload) > 15 * 1024 * 1024:
            raise ValueError("Model reference exceeds 15 MB.")
with Image.open(BytesIO(payload)) as im:
    if im.width * im.height > 30_000_000:
        raise ValueError("Model photograph exceeds 30 megapixels.")
    person = ImageOps.exif_transpose(im).convert("RGB").copy()

print(f"Female fashion-model catalogue reference: style {PERSON_REFERENCE_STYLE}")
print(f"Garment being applied: style {STYLE_CODE}")
print("Internal evaluation only: confirm appropriate model likeness/AI editing permissions.")
preview_scale = min(420 / person.width, 1)
display(person.resize((
    max(1, round(person.width * preview_scale)),
    max(1, round(person.height * preview_scale))
)))

#========================================================================
# AUTOMATED STEP 6 / 8
#========================================================================
# Run after Sections 2, 3, 4 and 5 have completed successfully.
import sys
from pathlib import Path
import torch
from IPython.display import display

ROOT = Path("/content")
REPO = ROOT / "fashn-vton-1.5"
WEIGHTS = REPO / "weights"
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Section 2 hasn't finished: run the installation cell first.")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
try:
    from fashn_vton import TryOnPipeline
except Exception as e:
    raise RuntimeError(
        "Cannot import fashn_vton. Run Section 2, confirm the green INSTALLATION VERIFIED message, "
        "and then retry. If imports still fail, restart session and Run all."
    ) from e

expected = [
    WEIGHTS / "model.safetensors",
    WEIGHTS / "dwpose" / "yolox_l.onnx",
    WEIGHTS / "dwpose" / "dw-ll_ucoco_384.onnx"
]
if any(not p.is_file() or p.stat().st_size == 0 for p in expected):
    raise RuntimeError("Section 3 is incomplete: download model weights first.")
if "person" not in globals() or "garment" not in globals():
    raise RuntimeError("Run Sections 4 and 5 to load the garment and the official demo person.")
if not torch.cuda.is_available():
    raise RuntimeError("No active GPU. Go to Runtime → Change runtime type → GPU.")

gpu_name = torch.cuda.get_device_name(0)
print("Using GPU:", gpu_name)
if not torch.cuda.is_bf16_supported():
    print("⚠️ This GPU does not support bf16, so the model will use float32 and may exceed VRAM.")
torch.cuda.empty_cache()
print("Initializing actual FASHN VTON v1.5 (research-only)...")
try:
    pipeline = TryOnPipeline(weights_dir=str(WEIGHTS), device="cuda")
    output = pipeline(
        person_image=person,
        garment_image=garment,
        category="one-pieces",
        garment_photo_type="model",
        num_samples=1,
        num_timesteps=20,
        seed=42,
        segmentation_free=True
    )
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    raise RuntimeError(
        f"{gpu_name} ran out of GPU memory. The free T4 may be unable to run this model "
        "because older GPUs use float32. No paid upgrade is required to understand the error."
    ) from e

result = output.images[0]
RESULT_PATH = ROOT / ("manisha_tryon_" + STYLE_CODE + "_research.png")
result.save(RESULT_PATH)
print("✅ AI TRY-ON GENERATION COMPLETED. Internal evaluation only.")
display(result)
print("Temporary result:", RESULT_PATH)

#========================================================================
# AUTOMATED STEP 7 / 8
#========================================================================
from IPython.display import display
print("Original catalogue garment:")
display(garment)
print("Female-model catalogue reference (existing Manisha Overseas photo):")
display(person)
print("AI-generated internal evaluation:")
display(result)

#========================================================================
# AUTOMATED STEP 8 / 8
#========================================================================
# Optional: uncomment only if you need a local copy for internal assessment.
# from google.colab import files
# files.download(str(RESULT_PATH))

print("Local result:", RESULT_PATH)
print("To remove the local result now, run:")
print("RESULT_PATH.unlink(missing_ok=True)")
print("Then use Runtime → Disconnect and delete runtime to discard the temporary VM.")